In [ ]:
#
# Project:
#      PyTorch Dojo (https://github.com/wo3kie/ml-dojo)
#
# Author:
#      Lukasz Czerwinski (https://www.lukaszczerwinski.pl/)
#

In [ ]:
import math
import torch


def _nop(*args, **kwargs):
    """Default no logging callback."""
    pass


def _default_logger(lhs, rhs, atol, rtol):
    """Default logging callback."""
    class _ansi:
        RED = "\033[31m"
        YELLOW = "\033[33m"
        RESET = "\033[0m"

    print(
        f"{_ansi.RED}[approx mismatch]{_ansi.RESET} "
        f"{_ansi.YELLOW}{lhs}{_ansi.RESET} vs "
        f"{_ansi.YELLOW}{rhs}{_ansi.RESET} "
        f"(atol={atol}, rtol={rtol})"
    )


class approx:
    def __init__(self, value, atol: float = 0.001, rtol: float = 0.0, callback=_nop):
        self._value = value
        self._atol = atol
        self._rtol = rtol
        self._callback = callback

    def log(self, callback=None):
        if callback is None:
            self._callback = _default_logger
        else:
            self._callback = callback
        return self

    def _to_tensor(self, x):
        if isinstance(x, torch.Tensor):
            return x
        
        return torch.tensor(x, dtype=torch.float32)

    def _compare(self, lhs, rhs, atol, rtol):
        if isinstance(lhs, torch.Tensor) or isinstance(rhs, torch.Tensor):
            lhs = self._to_tensor(lhs)
            rhs = self._to_tensor(rhs)
            return torch.allclose(lhs, rhs, atol=atol, rtol=rtol)

        return math.isclose(float(lhs), float(rhs), abs_tol=atol, rel_tol=rtol)

    def __eq__(self, other):
        if (ok := self._compare(self._value, other, self._atol, self._rtol)) == False:
            self._callback(self._value, other, self._atol, self._rtol)

        return ok

    def __req__(self, other):
        if (ok := self._compare(other, self._value, self._atol, self._rtol)) == False:
            self._callback(other, self._value, self._atol, self._rtol)

        return ok

    def __ne__(self, other):
        return not self.__eq__(other)

    def __rne__(self, other):
        return not self.__req__(other)
    

def test_basic():
    assert 1.0 == approx(1.0)
    assert approx(1.0) == 1.0

    assert 1.05 == approx(1.0, atol=0.1)
    assert approx(1.0, atol=0.1) == 1.05

    assert 105 == approx(100, rtol=0.1)
    assert approx(100, rtol=0.1) == 105


def test_logging():
    log_called = False
    def callback(lhs, rhs, atol, rtol):
        nonlocal log_called
        log_called = True

    1 == approx(2.0).log(callback)
    
    assert log_called


if __name__ == "__main__":
    test_basic()
    test_logging()
